# Практический кейс «Прогнозирование конечных свойств новых материалов (композиционных материалов)». 

***Задачи, решение которых отражено в данном ноутбуке:***
- Разведочный анализ;
- Предобработка данных;
- Обучение моделей ML.

***Цель работы:*** 
- разработать модели для прогноза модуля упругости при растяжении и прочности при растяжении

# Импорты

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LassoCV, ElasticNetCV, RidgeCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.svm import SVR

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout

import pickle

%matplotlib inline

In [ ]:
!python --version

In [ ]:
!jupyter notebook --version

### Версии библиотек

 - pandas: '2.1.4'
 - numpy: '1.26.4'
 - matplotlib: '3.8.0'
 - seaborn: '0.13.2'
 - scikit-learn: '1.5.1'
 - tensorflow: '2.17.0'

# Разведочный анализ данных

Загружаем таблицы формата Excel в два Пандас Датафрейма:

In [ ]:
url_bp = 'https://raw.githubusercontent.com/judeclapton/composite_materials/main/data/X_bp.xlsx'
bp_df = pd.read_excel(url_bp, engine='openpyxl', index_col=0)

In [ ]:
bp_df.head()

In [ ]:
url_nup = 'https://raw.githubusercontent.com/judeclapton/composite_materials/main/data/X_nup.xlsx'
nup_df = pd.read_excel(url_nup, engine='openpyxl', index_col=0)

In [ ]:
nup_df.head()

Проверим размерность в двух таблицах:

In [ ]:
print(f"Количество строк ({bp_df.shape[0]}) и признаков ({bp_df.shape[1]}) в X_bp dataframe")
print(f"Количество строк ({nup_df.shape[0]}) и признаков ({nup_df.shape[1]}) в X_nup dataframe")

*Второй датафрейм имеет больше значений. Во время объединения по индексу, 17 последних значений учтены не будут.*

In [ ]:
# Inner Join

df = bp_df.join(nup_df, how='inner')

In [ ]:
df.head()

Проверим размерность нового датафрейма:

In [ ]:
print(f"Количество строк ({df.shape[0]}) и признаков ({df.shape[1]}) в df dataframe")

Проверим типы признаков:

In [ ]:
df.dtypes

*Признаки не нуждаются в преобразовании, на данный момент, так как они все имеют числовой тип.*

Проверим на количество пропусков:

In [ ]:
df.isna().sum()

*Данные достаточно полные.*

Проверим на количество уникальных значений:

In [ ]:
df.nunique()

*Признак "Угол нашивки, град" имеет всего два уникальных значения и может быть преобразован в бинарный тип, если это будет полезно при дальнейшей работе.*

Посмотрим сводную статистику в нашем датафрейме, в более удобночитаемом виде:

In [ ]:
df.describe().T

Визуализируем распределение значений:

In [ ]:
df.hist(bins=15, figsize=(15, 10))
plt.show()

*Значения, в целом, нормально распределены, но для работы с ними потребуется нормализация/стандартизация.*

Масштабируем признаки, чтобы наглядно построить "ящики с усами" для каждой переменной:

In [ ]:
scaler = MinMaxScaler()
scaler.fit(df)

In [ ]:
plt.boxplot(pd.DataFrame(scaler.transform(df)), 
            labels=df.columns, 
            patch_artist=True, 
            meanline=True, 
            vert=False, 
            boxprops=dict(facecolor='seagreen', color='black'), 
            medianprops=dict(color='black'), 
            whiskerprops=dict(color='black'), 
            capprops=dict(color = 'black'), 
            flierprops=dict(marker='o', markerfacecolor='red', markersize=6, linestyle='none')
           )
plt.gca().invert_yaxis()
plt.show()

*Признаки имеют довольно много выбросов, но с этим мы поработаем на следующем этапе.*

Визуализируем общую корреляцию между признаками:

In [ ]:
matrix = np.triu(df.corr())
plt.figure(figsize=(15, 10))
sns.heatmap(df.corr(), annot = True, cmap= 'magma', linewidths=1, linecolor='white', mask = matrix)
plt.xticks(rotation = 45, ha = 'right')
plt.show()

*К сожалению,общих корреляций очень мало, что даёт понять, что будет сложно работать с представленными данными.*

# Предобработка данных

Разделим данные на обучающую и тестовую выборки:

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

Обработку выбросов мы будем делать при помощи метода межквартильного размаха. Подготовим метод:

In [ ]:
def iqr_method(df, columns):
    for column in columns:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    return df

## Работа с выбросами в обучающей выборке

Масштабируем обучающую выборку, для более удобной работы:

In [ ]:
scaler = MinMaxScaler()
scaler.fit(train_df)

Визуализируем и посчитаем выбросы в обучающей выборке:

In [ ]:
scaled_df = pd.DataFrame(scaler.transform(train_df), columns=train_df.columns)
outliers = {}
total = 0

for column in scaled_df:
    Q1 = scaled_df[column].quantile(0.25)
    Q3 = scaled_df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers[column] = ((scaled_df[column] < lower_bound) | (scaled_df[column] > upper_bound)).sum()
    total += outliers[column]

print(pd.DataFrame.from_dict(outliers, orient='index', columns=['Outliers']))
print("Общее количество возможных выбросов: ", total)

In [ ]:
plt.boxplot(pd.DataFrame(scaler.transform(train_df)), 
            labels=df.columns, 
            patch_artist=True, 
            meanline=True, 
            vert=False, 
            boxprops=dict(facecolor='seagreen', color='black'), 
            medianprops=dict(color='black'), 
            whiskerprops=dict(color='black'), 
            capprops=dict(color = 'black'), 
            flierprops=dict(marker='o', markerfacecolor='red', markersize=6, linestyle='none')
           )
plt.gca().invert_yaxis()
plt.show()

In [ ]:
print('Количество значений в признаках: ', train_df.shape[0])

Обработаем выбросы:

In [ ]:
train_df = iqr_method(train_df, train_df.columns)

In [ ]:
print('Количество значений в признаках: ', train_df.shape[0])

*Количество значений уменьшилось на **72 единицы**.*

Посмотрим на результаты после обработки:

In [ ]:
scaled_df = pd.DataFrame(scaler.transform(train_df), columns=train_df.columns)
outliers = {}
total = 0

for column in scaled_df:
    Q1 = scaled_df[column].quantile(0.25)
    Q3 = scaled_df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers[column] = ((scaled_df[column] < lower_bound) | (scaled_df[column] > upper_bound)).sum()
    total += outliers[column]

print(pd.DataFrame.from_dict(outliers, orient='index', columns=['Outlier Count']))
print("Общее количество возможных выбросов: ", total)

In [ ]:
plt.boxplot(pd.DataFrame(scaler.transform(train_df)), 
            labels=df.columns, 
            patch_artist=True, 
            meanline=True, 
            vert=False, 
            boxprops=dict(facecolor='seagreen', color='black'), 
            medianprops=dict(color='black'), 
            whiskerprops=dict(color='black'), 
            capprops=dict(color = 'black'), 
            flierprops=dict(marker='o', markerfacecolor='red', markersize=6, linestyle='none')
           )
plt.gca().invert_yaxis()
plt.show()

*Количество выбросов заметно уменьшилось.*

## Работа с выбросами в тестовой выборке

Масштабируем тестовую выборку, для более удобной работы:

In [ ]:
scaler = MinMaxScaler()
scaler.fit(test_df)

Визуализируем и посчитаем выбросы в тестовой выборке:

In [ ]:
scaled_df = pd.DataFrame(scaler.transform(test_df), columns=test_df.columns)
outliers = {}
total = 0

for column in scaled_df:
    Q1 = scaled_df[column].quantile(0.25)
    Q3 = scaled_df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers[column] = ((scaled_df[column] < lower_bound) | (scaled_df[column] > upper_bound)).sum()
    total += outliers[column]

print(pd.DataFrame.from_dict(outliers, orient='index', columns=['Outlier Count']))
print("Общее количество возможных выбросов: ", total)

In [ ]:
plt.boxplot(pd.DataFrame(scaler.transform(test_df)), 
            labels=df.columns, 
            patch_artist=True, 
            meanline=True, 
            vert=False, 
            boxprops=dict(facecolor='seagreen', color='black'), 
            medianprops=dict(color='black'), 
            whiskerprops=dict(color='black'), 
            capprops=dict(color = 'black'), 
            flierprops=dict(marker='o', markerfacecolor='red', markersize=6, linestyle='none')
           )
plt.gca().invert_yaxis()
plt.show()

*Количество выбросов не столь велико, поэтому пока попробуем поработать без обработки тестовых данных (так как это, в принципе, не обязательно).*

Нормализуем данные по параметрам обучающей выборки:

In [ ]:
scaler = MinMaxScaler()
scaler.fit(train_df)

train_df_norm = pd.DataFrame(scaler.transform(train_df), columns=train_df.columns)
test_df_norm = pd.DataFrame(scaler.transform(test_df), columns=test_df.columns)

# Машинное обучение

Разделим тренировочную выборку для модуля упругости при растяжении и прочности при растяжении:

In [ ]:
X_train, y_train_modulus, y_train_strength = (
    train_df_norm.drop(columns=['Модуль упругости при растяжении, ГПа', 'Прочность при растяжении, МПа']), 
    train_df_norm['Модуль упругости при растяжении, ГПа'], 
    train_df_norm['Прочность при растяжении, МПа']
)

Разделим тестовую выборку для модуля упругости при растяжении и прочности при растяжении:

In [ ]:
X_test, y_test_modulus, y_test_strength = (
    test_df_norm.drop(columns=['Модуль упругости при растяжении, ГПа', 'Прочность при растяжении, МПа']), 
    test_df_norm['Модуль упругости при растяжении, ГПа'], 
    test_df_norm['Прочность при растяжении, МПа']
)

Проверим размерности:

In [ ]:
print("X_train: ", X_train.shape, ", y_train_modulus: ", y_train_modulus.shape, ", y_train_strength: ", y_train_strength.shape)
print("X_test: ", X_test.shape, ", y_test_modulus: ", y_test_modulus.shape, ", y_test_strength: ", y_test_strength.shape)

Подготовим также тренировочную и тестовую выбоки соотношения матрица-наполнитель:

In [ ]:
X_train_matrix, y_train_matrix = train_df_norm.drop(columns='Соотношение матрица-наполнитель'
), train_df_norm['Соотношение матрица-наполнитель']

X_test_matrix, y_test_matrix = test_df_norm.drop(columns='Соотношение матрица-наполнитель'
), test_df_norm['Соотношение матрица-наполнитель']

Проверим размерности:

In [ ]:
print('X_train_matrix: ', X_train_matrix.shape, 'y_train_matrix: ', y_train_matrix.shape)
print('X_test_matrix: ', X_test_matrix.shape, 'y_test_matrix: ', y_test_matrix.shape)

## Алгоритм машинного обучения для определения значений модуля упругости при растяжении и прочности при растяжении

Визуализируем общую корреляцию признаков, после всех наших прошлых действий:

In [ ]:
matrix = np.triu(train_df_norm.corr())
plt.figure(figsize=(15, 10))
sns.heatmap(train_df_norm.corr(), annot = True, cmap= 'magma', linewidths=1, linecolor='white', mask = matrix)
plt.xticks(rotation = 45, ha = 'right')
plt.show()

In [ ]:
correlations = train_df_norm.corr()

Посмотрим корреляцию признаков с модулем упругости при растяжении:

In [ ]:
print(correlations['Модуль упругости при растяжении, ГПа'].sort_values(ascending=False))

А теперь корреляцию признаков с прочностью при растяжении:

In [ ]:
print(correlations['Прочность при растяжении, МПа'].sort_values(ascending=False))

*К сожалению, корреляции очень слабы.*

Попробуем найти общие корреляции для обоих признаков, в качестве минимального порога возьмём минимальную корреляцию из двух таблиц выше: 

In [ ]:
correlation_threshold = 0.001326
correlation_matrix = train_df_norm.corr()
target_corr = correlation_matrix[['Модуль упругости при растяжении, ГПа', 'Прочность при растяжении, МПа']]

useful_features = target_corr[
    (target_corr['Модуль упругости при растяжении, ГПа'] > correlation_threshold ) &
    (target_corr['Прочность при растяжении, МПа'] > correlation_threshold)
]

if useful_features.empty:
    print("Нет признаков с корреляцией выше заданного порога для обоих целевых признаков.")
else:
    plt.figure(figsize=(8, 6))
    sns.heatmap(useful_features.T, annot=True, cmap='winter', linewidths=1, linecolor='white', fmt=".2f",
                annot_kws={"size": 10})
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(rotation=0, fontsize=12)
    plt.show()

*Наилучшим образом здесь себя показывают признаки "Модуль упругости, ГПа" и "Потребление смолы, г/м2", но корреляция всё равно очень и очень слаба. То есть зависимости признаков с целевыми переменными практически нет.*

Визуализируем графики ядерной оценки плотности признаков до нормализации и после:

In [ ]:
fig, ax = plt.subplots(figsize = (18, 5))
train_df.plot(kind = 'kde', ax = ax)
plt.title('Ядерная оценка плотности до нормализации')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (18, 5))
train_df_norm.plot(kind = 'kde', ax = ax)
plt.title('Ядерная оценка плотности после нормализации')
plt.show()

*При сравнении двух графиков видно, что нормализация прошла довольно успешно. В связи с малым количеством значений и слабой корреляцией целевых переменных и остальных признаков, мало что можно сделать ещё. Попробуем перейти наконец к непосредственной задаче.*

### Первая гипотеза

Итак, наша первая гипотеза будет звучать так:    
*Несмотря на слабую корреляцию и малое количество изначальных данных, нам должно быть их достаточно для построения модели машинного обучения.*

#### Обучение

Для начала создадим таблицу в которую будем складывать различные метрики для работы каждой модели:

In [ ]:
metrics_modulus = ['MSEm', 'MAEm', 'R²m train', 'R²m test']
metrics_strength = ['MSEs', 'MAEs', 'R²s train', 'R²s test']
models = ['RandomForestRegressor', 'LinearRegression', 'LassoCV', 'ElasticNetCV', 'RidgeCV', 'KNeighborsRegressor', 
         'GradientBoostingRegressor', 'SVR']

ms_table = pd.DataFrame(index=models, columns=metrics_modulus + metrics_strength)

In [ ]:
ms_table

Затем создадим специальные методы, чтобы проще было конструировать разные модели:

In [ ]:
def create_modulus_model(model, model_name):
    model.fit(X_train, y_train_modulus)
    y_pred_m = model.predict(X_test)
    
    # средняя квадратичная ошибка:
    mse = mean_squared_error(y_test_modulus, y_pred_m)
    ms_table.loc[model_name, 'MSEm'] = round(mse, 3)
    print("MSE для модуля упругости:", mse)
    
    #средняя абсолютная ошибка:
    mae = mean_absolute_error(y_test_modulus, y_pred_m)
    ms_table.loc[model_name, 'MAEm'] = round(mae, 3)
    print("MAE для модуля упругости:", mae)
    
    # коэффициент детерминации для обучающей выборки:
    r2_train = model.score(X_train, y_train_modulus)
    ms_table.loc[model_name, 'R²m train'] = round(r2_train, 3)
    print("R² на обучающей выборке для модуля упругости:", r2_train)
    
    # коэффициент детерминации для тестовой выборки:
    r2_test = r2_score(y_test_modulus, y_pred_m)
    ms_table.loc[model_name, 'R²m test'] = round(r2_test, 3)
    print("R² на тестовой выборке для модуля упругости:", r2_test)


def create_strength_model(model, model_name):
    model.fit(X_train, y_train_strength)
    y_pred_s = model.predict(X_test)
    
    # средняя квадратичная ошибка:
    mse = mean_squared_error(y_test_strength, y_pred_s)
    ms_table.loc[model_name, 'MSEs'] = round(mse, 3)
    print("MSE для прочности:", mse)
    
    #средняя абсолютная ошибка:
    mae = mean_absolute_error(y_test_strength, y_pred_s)
    ms_table.loc[model_name, 'MAEs'] = round(mae, 3)
    print("MAE для прочности:", mae)
    
    # коэффициент детерминации для обучающей выборки:
    r2_train = model.score(X_train, y_train_strength)
    ms_table.loc[model_name, 'R²s train'] = round(r2_train, 3)
    print("R² на обучающей выборке для прочности:", r2_train)
    
    # коэффициент детерминации для тестовой выборки:
    r2_test = r2_score(y_test_strength, y_pred_s)
    ms_table.loc[model_name, 'R²s test'] = round(r2_test, 3)
    print("R² на тестовой выборке для прочности:", r2_test)
    

##### `RandomForestRegressor`:

Для начала подберём подходящие значения n_estimators и max_depth для моделей Случайного Леса:

In [ ]:
various = {'n_estimators': [100, 200], 
           'max_depth': [5, 10, 15, 20]}

results_m = GridSearchCV(RandomForestRegressor(), various, cv=9, n_jobs=-1, verbose=1)
results_m.fit(X_train, y_train_modulus)
print("Лучшее значение n_estimators для модуля упругости: ", results_m.best_params_)

results_s = GridSearchCV(RandomForestRegressor(), various, cv=9, n_jobs=-1, verbose=1)
results_s.fit(X_train, y_train_strength)
print("Лучшее значение n_estimators для прочности: ", results_s.best_params_)

In [ ]:
model_rf = RandomForestRegressor(max_depth=5, n_estimators=200)

In [ ]:
create_modulus_model(model_rf, "RandomForestRegressor")

In [ ]:
create_strength_model(model_rf, "RandomForestRegressor")

##### `LinearRegression`:

In [ ]:
model_lr = LinearRegression()

In [ ]:
create_modulus_model(model_lr, "LinearRegression")

In [ ]:
create_strength_model(model_lr, "LinearRegression")

##### `LassoCV`

In [ ]:
model_lasso = LassoCV()

In [ ]:
create_modulus_model(model_lasso, "LassoCV")

In [ ]:
create_strength_model(model_lasso, "LassoCV")

##### `ElasticNetCV`

In [ ]:
model_en = ElasticNetCV()

In [ ]:
create_modulus_model(model_en, "ElasticNetCV")

In [ ]:
create_strength_model(model_en, "ElasticNetCV")

##### `RidgeCV`

In [ ]:
model_ridge = RidgeCV()

In [ ]:
create_modulus_model(model_ridge, "RidgeCV")

In [ ]:
create_strength_model(model_ridge, "RidgeCV")

##### `KNeighborsRegressor`

Для начала подберём подходящие значения n_neighbors для моделей KNN:

In [ ]:
various = {'n_neighbors': range(1, 501)}

results_m = GridSearchCV(KNeighborsRegressor(), various, cv=9, n_jobs=-1, verbose=1)
results_m.fit(X_train, y_train_modulus)
print("Лучшее значение n_neighbors для модуля упругости: ", results_m.best_params_)

results_s = GridSearchCV(KNeighborsRegressor(), various, cv=9, n_jobs=-1, verbose=1)
results_s.fit(X_train, y_train_strength)
print("Лучшее значение n_neighbors для прочности: ", results_s.best_params_)

In [ ]:
model_knn_m = KNeighborsRegressor(n_neighbors=206)
model_knn_s = KNeighborsRegressor(n_neighbors=434)

In [ ]:
create_modulus_model(model_knn_m, "KNeighborsRegressor")

In [ ]:
create_strength_model(model_knn_s, "KNeighborsRegressor")

##### `GradientBoostingRegressor`

Для начала подберём подходящие значения n_estimators, max_depth и learning_rate для градиентного бустинга:

In [ ]:
various = {
    'n_estimators': [100, 200], 
    'max_depth': [3, 5, 7], 
    'learning_rate': [0.01, 0.05, 0.1]
}

results_m = GridSearchCV(GradientBoostingRegressor(), various, cv=9, n_jobs=-1, verbose=1)
results_m.fit(X_train, y_train_modulus)
print("Лучшие значения для модуля упругости: ", results_m.best_params_)

results_s = GridSearchCV(GradientBoostingRegressor(), various, cv=9, n_jobs=-1, verbose=1)
results_s.fit(X_train, y_train_strength)
print("Лучшие значения для прочности: ", results_s.best_params_)

In [ ]:
model_gb = GradientBoostingRegressor(learning_rate=0.01, max_depth=3, n_estimators=100)

In [ ]:
create_modulus_model(model_gb, "GradientBoostingRegressor")

In [ ]:
create_strength_model(model_gb, "GradientBoostingRegressor")

##### `SVR`

Для начала подберём подходящие значения `kernel` и `C` для моделей метода опорных векторов:

In [ ]:
various = {'C': np.logspace(-3, 2, 6), 'kernel': ['linear', 'poly', 'rbf', 'sigmoid']}

results_m = GridSearchCV(SVR(), various, cv=9, n_jobs=-1, verbose=1)
results_m.fit(X_train, y_train_modulus)
print("Лучшие значения для модуля упругости: ", results_m.best_params_)

results_s = GridSearchCV(SVR(), various, cv=9, n_jobs=-1, verbose=1)
results_s.fit(X_train, y_train_strength)
print("Лучшие значения для прочности: ", results_s.best_params_)

In [ ]:
model_svr_m = SVR(kernel='rbf', C=0.01)
model_svr_s = SVR(kernel='sigmoid', C=0.1)

In [ ]:
create_modulus_model(model_svr_m, "SVR")

In [ ]:
create_strength_model(model_svr_s, "SVR")

Отсортируем таблицу по результатам коэффициентов детерминации (в тестовой выборке) для обеих целевых переменных:

In [ ]:
ms_table.sort_values(['R²m test', 'R²s test'], ascending=False)

***Вывод:***    
Метрики очень малы; коэффициент детерминации очень мал или даже отрицателен, а это значит, что модели не научились правильно предсказывать целевые значения.

### Вторая гипотеза

Так как результаты работы по первой гипотезе весьма сложно назвать хоть сколько-нибудь подходящими, то выдвинем следующую:    
*Данные содержат много неверных вводных, поэтому тестовые данные всё же надо очистить от выбросов*.    
Посмотрим что получится.

До этого мы остановились на подсчёте и визуализации выбросов в тестовой выборке. Повторим эти шаги:

In [ ]:
scaler = MinMaxScaler()
scaler.fit(test_df)

In [ ]:
scaled_df = pd.DataFrame(scaler.transform(test_df), columns=test_df.columns)
outliers = {}
total = 0

for column in scaled_df:
    Q1 = scaled_df[column].quantile(0.25)
    Q3 = scaled_df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers[column] = ((scaled_df[column] < lower_bound) | (scaled_df[column] > upper_bound)).sum()
    total += outliers[column]

print(pd.DataFrame.from_dict(outliers, orient='index', columns=['Outlier Count']))
print("Общее количество возможных выбросов: ", total)

In [ ]:
plt.boxplot(pd.DataFrame(scaler.transform(test_df)), 
            labels=df.columns, 
            patch_artist=True, 
            meanline=True, 
            vert=False, 
            boxprops=dict(facecolor='seagreen', color='black'), 
            medianprops=dict(color='black'), 
            whiskerprops=dict(color='black'), 
            capprops=dict(color = 'black'), 
            flierprops=dict(marker='o', markerfacecolor='red', markersize=6, linestyle='none')
           )
plt.gca().invert_yaxis()
plt.show()

In [ ]:
print('Количество значений в признаках: ', test_df.shape[0])

Обработаем выбросы в нашей тестовой выборке:

In [ ]:
test_df = iqr_method(test_df, test_df.columns)

In [ ]:
print('Количество значений в признаках: ', test_df.shape[0])

*Количество значений уменьшилось на **18 единиц**.*

Посмотрим на результаты после обработки:

In [ ]:
scaled_df = pd.DataFrame(scaler.transform(test_df), columns=test_df.columns)
outliers = {}
total = 0

for column in scaled_df:
    Q1 = scaled_df[column].quantile(0.25)
    Q3 = scaled_df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers[column] = ((scaled_df[column] < lower_bound) | (scaled_df[column] > upper_bound)).sum()
    total += outliers[column]

print(pd.DataFrame.from_dict(outliers, orient='index', columns=['Outlier Count']))
print("Общее количество возможных выбросов: ", total)

In [ ]:
plt.boxplot(pd.DataFrame(scaler.transform(test_df)), 
            labels=df.columns, 
            patch_artist=True, 
            meanline=True, 
            vert=False, 
            boxprops=dict(facecolor='seagreen', color='black'), 
            medianprops=dict(color='black'), 
            whiskerprops=dict(color='black'), 
            capprops=dict(color = 'black'), 
            flierprops=dict(marker='o', markerfacecolor='red', markersize=6, linestyle='none')
           )
plt.gca().invert_yaxis()
plt.show()

Тестовая выборка готова к дальнейшей работе.

Нормализуем данные по параметрам обучающей выборки:

In [ ]:
scaler = MinMaxScaler()
scaler.fit(train_df)

train_df_norm = pd.DataFrame(scaler.transform(train_df), columns=train_df.columns)
test_df_norm = pd.DataFrame(scaler.transform(test_df), columns=test_df.columns)

Обучающая выборка у нас уже была разделена ранее и в ней ничего не менялось. Поэтому разделим только тестовую:

In [ ]:
X_test, y_test_modulus, y_test_strength = (
    test_df_norm.drop(columns=['Модуль упругости при растяжении, ГПа', 'Прочность при растяжении, МПа']), 
    test_df_norm['Модуль упругости при растяжении, ГПа'], 
    test_df_norm['Прочность при растяжении, МПа']
)

Проверим размерности:

In [ ]:
print("X_test: ", X_test.shape, ", y_test_modulus: ", y_test_modulus.shape, ", y_test_strength: ", y_test_strength.shape)

In [ ]:
correlations = test_df_norm.corr()

Посмотрим корреляцию признаков с модулем упругости при растяжении:

In [ ]:
print(correlations['Модуль упругости при растяжении, ГПа'].sort_values(ascending=False))

А теперь корреляцию признаков с прочностью при растяжении:

In [ ]:
print(correlations['Прочность при растяжении, МПа'].sort_values(ascending=False))

*В целом, выглядит неплохо, но корреляции всё ещё весбма слабы.*

Попробуем найти общие корреляции для обоих признаков, в качестве минимального порога возьмём минимальную корреляцию из двух таблиц выше:

In [ ]:
correlation_threshold = 0.005753
correlation_matrix = test_df_norm.corr()
target_corr = correlation_matrix[['Модуль упругости при растяжении, ГПа', 'Прочность при растяжении, МПа']]

useful_features = target_corr[
    (target_corr['Модуль упругости при растяжении, ГПа'] > correlation_threshold ) &
    (target_corr['Прочность при растяжении, МПа'] > correlation_threshold)
]

if useful_features.empty:
    print("Нет признаков с корреляцией выше заданного порога для обоих целевых признаков.")
else:
    plt.figure(figsize=(8, 6))
    sns.heatmap(useful_features.T, annot=True, cmap='winter', linewidths=1, linecolor='white', fmt=".2f",
                annot_kws={"size": 10})
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(rotation=0, fontsize=12)
    plt.show()

*Общие взаимосвязи гораздо слабее, чем в данных обучающей выборки.*

Визуализируем графики ядерной оценки плотности признаков до нормализации и после:

In [ ]:
fig, ax = plt.subplots(figsize = (18, 5))
test_df.plot(kind = 'kde', ax = ax)
plt.title('Ядерная оценка плотности до нормализации')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (18, 5))
test_df_norm.plot(kind = 'kde', ax = ax)
plt.title('Ядерная оценка плотности до нормализации')
plt.show()

*Нормализовать обучающие данные получилось чуть лучше.*

#### Обучение #2

Снова создадим таблицу (к её названию добавим двойку) в которую будем складывать различные метрики для работы каждой модели:

In [ ]:
metrics_modulus = ['MSEm', 'MAEm', 'R²m train', 'R²m test']
metrics_strength = ['MSEs', 'MAEs', 'R²s train', 'R²s test']
models = ['RandomForestRegressor', 'LinearRegression', 'LassoCV', 'ElasticNetCV', 'RidgeCV', 'KNeighborsRegressor', 
         'GradientBoostingRegressor', 'SVR']

ms_table_2 = pd.DataFrame(index=models, columns=metrics_modulus + metrics_strength)

Переопределим метод, чтобы складывать новые данные именно в новую таблицу:

In [ ]:
def create_modulus_model(model, model_name):
    model.fit(X_train, y_train_modulus)
    y_pred_m = model.predict(X_test)
    
    # средняя квадратичная ошибка:
    mse = mean_squared_error(y_test_modulus, y_pred_m)
    ms_table_2.loc[model_name, 'MSEm'] = round(mse, 3)
    print("MSE для модуля упругости:", mse)
    
    #средняя абсолютная ошибка:
    mae = mean_absolute_error(y_test_modulus, y_pred_m)
    ms_table_2.loc[model_name, 'MAEm'] = round(mae, 3)
    print("MAE для модуля упругости:", mae)
    
    # коэффициент детерминации для обучающей выборки:
    r2_train = model.score(X_train, y_train_modulus)
    ms_table_2.loc[model_name, 'R²m train'] = round(r2_train, 3)
    print("R² на обучающей выборке для модуля упругости:", r2_train)
    
    # коэффициент детерминации для тестовой выборки:
    r2_test = r2_score(y_test_modulus, y_pred_m)
    ms_table_2.loc[model_name, 'R²m test'] = round(r2_test, 3)
    print("R² на тестовой выборке для модуля упругости:", r2_test)


def create_strength_model(model, model_name):
    model.fit(X_train, y_train_strength)
    y_pred_s = model.predict(X_test)
    
    # средняя квадратичная ошибка:
    mse = mean_squared_error(y_test_strength, y_pred_s)
    ms_table_2.loc[model_name, 'MSEs'] = round(mse, 3)
    print("MSE для прочности:", mse)
    
    #средняя абсолютная ошибка:
    mae = mean_absolute_error(y_test_strength, y_pred_s)
    ms_table_2.loc[model_name, 'MAEs'] = round(mae, 3)
    print("MAE для прочности:", mae)
    
    # коэффициент детерминации для обучающей выборки:
    r2_train = model.score(X_train, y_train_strength)
    ms_table_2.loc[model_name, 'R²s train'] = round(r2_train, 3)
    print("R² на обучающей выборке для прочности:", r2_train)
    
    # коэффициент детерминации для тестовой выборки:
    r2_test = r2_score(y_test_strength, y_pred_s)
    ms_table_2.loc[model_name, 'R²s test'] = round(r2_test, 3)
    print("R² на тестовой выборке для прочности:", r2_test)

Обучающая выборка у нас не менялась, поэтому нет смысла заново учить модели. Но у нас изменились тестовые данные, поэтому проверим уже готовые модели на работу с ними.

##### `RandomForestRegressor`:

In [ ]:
create_modulus_model(model_rf, "RandomForestRegressor")

In [ ]:
create_strength_model(model_rf, "RandomForestRegressor")

##### `LinearRegression`:

In [ ]:
create_modulus_model(model_lr, "LinearRegression")

In [ ]:
create_strength_model(model_lr, "LinearRegression")

##### `LassoCV`

In [ ]:
create_modulus_model(model_lasso, "LassoCV")

In [ ]:
create_strength_model(model_lasso, "LassoCV")

##### `ElasticNetCV`

In [ ]:
create_modulus_model(model_en, "ElasticNetCV")

In [ ]:
create_strength_model(model_en, "ElasticNetCV")

##### `RidgeCV`

In [ ]:
create_modulus_model(model_ridge, "RidgeCV")

In [ ]:
create_strength_model(model_ridge, "RidgeCV")

##### `KNeighborsRegressor`

In [ ]:
create_modulus_model(model_knn_m, "KNeighborsRegressor")

In [ ]:
create_strength_model(model_knn_s, "KNeighborsRegressor")

##### `GradientBoostingRegressor`

In [ ]:
create_modulus_model(model_gb, "GradientBoostingRegressor")

In [ ]:
create_strength_model(model_gb, "GradientBoostingRegressor")

##### `SVR`

In [ ]:
create_modulus_model(model_svr_m, "SVR")

In [ ]:
create_strength_model(model_svr_s, "SVR")

Отсортируем таблицу по результатам коэффициентов детерминации (в тестовой выборке) для обеих целевых переменных:

In [ ]:
ms_table_2.sort_values(['R²m test', 'R²s test'], ascending=False)

Выведем также таблицу из первой гипотезы:

In [ ]:
ms_table.sort_values(['R²m test', 'R²s test'], ascending=False)

Значения метрик для моделей не особо изменились. Коэффициет всё также мал и признаки всё также мало как помогают моделям уловить взимосвязи.

***Вывод:***    
В сравнении с первой гипотезой, мы никуда особо не продвинулись. Предсказать целевые значения модели всё также не могут.

**ОБНОВЛЕНИЕ:** сохраним модель LinearRegression для прочности при растяжении, так как в конце исследования становится ясно, что это лучшая модель по показателям.

In [ ]:
model_lr.fit(X_train, y_train_strength)

In [ ]:
pickle.dump({'model': model_lr, 'scaler': scaler}, open("./models/strength_model_with_scaler.pkl", 'wb'))

In [ ]:
class StrengthModelWithScaler:
    def __init__(self, model, scaler):
        self.model = model
        self.scaler = scaler

    def predict(self, X):
        X_full = X.copy()
        
        missing_columns = ['Модуль упругости при растяжении, ГПа', 'Прочность при растяжении, МПа']
        for col in missing_columns:
            if col not in X_full.columns:
                X_full[col] = 0

        X_denorm = self.scaler.inverse_transform(X_full)
        X_denorm_df = pd.DataFrame(X_denorm, columns=X_full.columns)

        X_denorm_df = X_denorm_df.drop(columns=missing_columns, errors='ignore')

        y_pred = self.model.predict(X_denorm_df)
        return y_pred.flatten()

In [ ]:
loaded_data = pickle.load(open("./models/strength_model_with_scaler.pkl", 'rb'))

In [ ]:
strength_model = StrengthModelWithScaler(loaded_data['model'], loaded_data['scaler'])

In [ ]:
prediction = strength_model.predict(X_test)

In [ ]:
prediction[0]

Модель рабочая, хоть предсказание и так себе.

### Третья гипотеза

Признаки, которые весьма слабо коррелируют с целевыми значениями или не коррелируют совсем, только мешают обучению моделей и, соответственно, их предсказательной способности. Поэтому стоит оставить только те переменные, которые имеют хоть какую-то положительную связь с модулем упругости и прочностью.

Ещё раз посмотрим с какими признаками коррелируют целевые значения в обучающей и тестовой выборках.

In [ ]:
correlation_threshold = 0.001326
correlation_matrix = train_df.corr()
target_corr = correlation_matrix[['Модуль упругости при растяжении, ГПа', 'Прочность при растяжении, МПа']]

useful_features = target_corr[
    (target_corr['Модуль упругости при растяжении, ГПа'] > correlation_threshold ) &
    (target_corr['Прочность при растяжении, МПа'] > correlation_threshold)
]

if useful_features.empty:
    print("Нет признаков с корреляцией выше заданного порога для обоих целевых признаков.")
else:
    plt.figure(figsize=(8, 6))
    sns.heatmap(useful_features.T, annot=True, cmap='winter', linewidths=1, linecolor='white', fmt=".2f",
                annot_kws={"size": 10})
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(rotation=0, fontsize=12)
    plt.show()

In [ ]:
correlation_threshold = 0.005753
correlation_matrix = test_df.corr()
target_corr = correlation_matrix[['Модуль упругости при растяжении, ГПа', 'Прочность при растяжении, МПа']]

useful_features = target_corr[
    (target_corr['Модуль упругости при растяжении, ГПа'] > correlation_threshold ) &
    (target_corr['Прочность при растяжении, МПа'] > correlation_threshold)
]

if useful_features.empty:
    print("Нет признаков с корреляцией выше заданного порога для обоих целевых признаков.")
else:
    plt.figure(figsize=(8, 6))
    sns.heatmap(useful_features.T, annot=True, cmap='winter', linewidths=1, linecolor='white', fmt=".2f",
                annot_kws={"size": 10})
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(rotation=0, fontsize=12)
    plt.show()

*В наших выборках только три признака совпадают в плане наличия корреляции (очень маленькой!) с целевыми значениями: "Модуль упругости, ГПА", "Потребление смолы, г/м2" и "Угол нашивки, град".*

Данные уже были нормализованы ранее, поэтому сразу переходим к разделению выборок, оставляя только нужные нам данные, которые мы определили выше.

Разделим тренировочную выборку для модуля упругости при растяжении и прочности при растяжении:

In [ ]:
X_train, y_train_modulus, y_train_strength = (
    train_df_norm[['Модуль упругости, ГПа', 'Потребление смолы, г/м2', 'Угол нашивки, град']], 
    train_df_norm['Модуль упругости при растяжении, ГПа'], 
    train_df_norm['Прочность при растяжении, МПа']
)

Разделим тестовую выборку для модуля упругости при растяжении и прочности при растяжении:

In [ ]:
X_test, y_test_modulus, y_test_strength = (
    test_df_norm[['Модуль упругости, ГПа', 'Потребление смолы, г/м2', 'Угол нашивки, град']], 
    test_df_norm['Модуль упругости при растяжении, ГПа'], 
    test_df_norm['Прочность при растяжении, МПа']
)

Проверим размерности:

In [ ]:
print("X_train: ", X_train.shape, ", y_train_modulus: ", y_train_modulus.shape, ", y_train_strength: ", y_train_strength.shape)
print("X_test: ", X_test.shape, ", y_test_modulus: ", y_test_modulus.shape, ", y_test_strength: ", y_test_strength.shape)

#### Обучение #3

И снова шаг за шагом повторим наши действия из прошлых гипотез. Но в отличие от второй попытки, здесь мы будем обучать модели заново, так как признаки в выборках изменились. 

Создадим таблицу под номером три:

In [ ]:
metrics_modulus = ['MSEm', 'MAEm', 'R²m train', 'R²m test']
metrics_strength = ['MSEs', 'MAEs', 'R²s train', 'R²s test']
models = ['RandomForestRegressor', 'LinearRegression', 'LassoCV', 'ElasticNetCV', 'RidgeCV', 'KNeighborsRegressor', 
         'GradientBoostingRegressor', 'SVR']

ms_table_3 = pd.DataFrame(index=models, columns=metrics_modulus + metrics_strength)

Снова переопределим метод:

In [ ]:
def create_modulus_model(model, model_name):
    model.fit(X_train, y_train_modulus)
    y_pred_m = model.predict(X_test)
    
    # средняя квадратичная ошибка:
    mse = mean_squared_error(y_test_modulus, y_pred_m)
    ms_table_3.loc[model_name, 'MSEm'] = round(mse, 3)
    print("MSE для модуля упругости:", mse)
    
    #средняя абсолютная ошибка:
    mae = mean_absolute_error(y_test_modulus, y_pred_m)
    ms_table_3.loc[model_name, 'MAEm'] = round(mae, 3)
    print("MAE для модуля упругости:", mae)
    
    # коэффициент детерминации для обучающей выборки:
    r2_train = model.score(X_train, y_train_modulus)
    ms_table_3.loc[model_name, 'R²m train'] = round(r2_train, 3)
    print("R² на обучающей выборке для модуля упругости:", r2_train)
    
    # коэффициент детерминации для тестовой выборки:
    r2_test = r2_score(y_test_modulus, y_pred_m)
    ms_table_3.loc[model_name, 'R²m test'] = round(r2_test, 3)
    print("R² на тестовой выборке для модуля упругости:", r2_test)


def create_strength_model(model, model_name):
    model.fit(X_train, y_train_strength)
    y_pred_s = model.predict(X_test)
    
    # средняя квадратичная ошибка:
    mse = mean_squared_error(y_test_strength, y_pred_s)
    ms_table_3.loc[model_name, 'MSEs'] = round(mse, 3)
    print("MSE для прочности:", mse)
    
    #средняя абсолютная ошибка:
    mae = mean_absolute_error(y_test_strength, y_pred_s)
    ms_table_3.loc[model_name, 'MAEs'] = round(mae, 3)
    print("MAE для прочности:", mae)
    
    # коэффициент детерминации для обучающей выборки:
    r2_train = model.score(X_train, y_train_strength)
    ms_table_3.loc[model_name, 'R²s train'] = round(r2_train, 3)
    print("R² на обучающей выборке для прочности:", r2_train)
    
    # коэффициент детерминации для тестовой выборки:
    r2_test = r2_score(y_test_strength, y_pred_s)
    ms_table_3.loc[model_name, 'R²s test'] = round(r2_test, 3)
    print("R² на тестовой выборке для прочности:", r2_test)

И приступим к обучению моделей!

##### `RandomForestRegressor:`

Для начала подберём подходящие значения n_estimators и max_depth для моделей Случайного Леса:

In [ ]:
various = {'n_estimators': [100, 200], 
           'max_depth': [5, 10, 15, 20]}

results_m = GridSearchCV(RandomForestRegressor(), various, cv=9, n_jobs=-1, verbose=1)
results_m.fit(X_train, y_train_modulus)
print("Лучшее значение n_estimators для модуля упругости: ", results_m.best_params_)

results_s = GridSearchCV(RandomForestRegressor(), various, cv=9, n_jobs=-1, verbose=1)
results_s.fit(X_train, y_train_strength)
print("Лучшее значение n_estimators для прочности: ", results_s.best_params_)

In [ ]:
model_rf_m = RandomForestRegressor(max_depth=5, n_estimators=200)
model_rf_s = RandomForestRegressor(max_depth=5, n_estimators=100)

In [ ]:
create_modulus_model(model_rf_m, "RandomForestRegressor")

In [ ]:
create_strength_model(model_rf_s, "RandomForestRegressor")

##### `LinearRegression`:

In [ ]:
model_lr = LinearRegression()

In [ ]:
create_modulus_model(model_lr, "LinearRegression")

In [ ]:
create_strength_model(model_lr, "LinearRegression")

##### `LassoCV`:

In [ ]:
model_lasso = LassoCV()

In [ ]:
create_modulus_model(model_lasso, "LassoCV")

In [ ]:
create_strength_model(model_lasso, "LassoCV")

##### `ElasticNetCV`:

In [ ]:
model_en = ElasticNetCV()

In [ ]:
create_modulus_model(model_en, "ElasticNetCV")

In [ ]:
create_strength_model(model_en, "ElasticNetCV")

##### `RidgeCV`:

In [ ]:
model_ridge = RidgeCV()

In [ ]:
create_modulus_model(model_ridge, "RidgeCV")

In [ ]:
create_strength_model(model_ridge, "RidgeCV")

##### `KNeighborsRegressor`:

Для начала подберём подходящие значения n_neighbors для моделей KNN:

In [ ]:
various = {'n_neighbors': range(1, 501)}

results_m = GridSearchCV(KNeighborsRegressor(), various, cv=9, n_jobs=-1, verbose=1)
results_m.fit(X_train, y_train_modulus)
print("Лучшее значение n_neighbors для модуля упругости: ", results_m.best_params_)

results_s = GridSearchCV(KNeighborsRegressor(), various, cv=9, n_jobs=-1, verbose=1)
results_s.fit(X_train, y_train_strength)
print("Лучшее значение n_neighbors для прочности: ", results_s.best_params_)

In [ ]:
model_knn_m = KNeighborsRegressor(n_neighbors=318)
model_knn_s = KNeighborsRegressor(n_neighbors=425)

In [ ]:
create_modulus_model(model_knn_m, "KNeighborsRegressor")

In [ ]:
create_strength_model(model_knn_s, "KNeighborsRegressor")

##### `GradientBoostingRegressor`:

Для начала подберём подходящие значения n_estimators, max_depth и learning_rate для градиентного бустинга:

In [ ]:
various = {
    'n_estimators': [100, 200], 
    'max_depth': [3, 5, 7], 
    'learning_rate': [0.01, 0.05, 0.1]
}

results_m = GridSearchCV(GradientBoostingRegressor(), various, cv=9, n_jobs=-1, verbose=1)
results_m.fit(X_train, y_train_modulus)
print("Лучшие значения для модуля упругости: ", results_m.best_params_)

results_s = GridSearchCV(GradientBoostingRegressor(), various, cv=9, n_jobs=-1, verbose=1)
results_s.fit(X_train, y_train_strength)
print("Лучшие значения для прочности: ", results_s.best_params_)

In [ ]:
model_gb_m = GradientBoostingRegressor(learning_rate=0.01, max_depth=3, n_estimators=100)
model_gb_s = GradientBoostingRegressor(learning_rate=0.01, max_depth=5, n_estimators=100)

In [ ]:
create_modulus_model(model_gb_m, "GradientBoostingRegressor")

In [ ]:
create_strength_model(model_gb_s, "GradientBoostingRegressor")

##### `SVR`:

Для начала подберём подходящие значения kernel и C для моделей метода опорных векторов:

In [ ]:
various = {'C': np.logspace(-3, 2, 6), 'kernel': ['linear', 'poly', 'rbf', 'sigmoid']}

results_m = GridSearchCV(SVR(), various, cv=9, n_jobs=-1, verbose=1)
results_m.fit(X_train, y_train_modulus)
print("Лучшие значения для модуля упругости: ", results_m.best_params_)

results_s = GridSearchCV(SVR(), various, cv=9, n_jobs=-1, verbose=1)
results_s.fit(X_train, y_train_strength)
print("Лучшие значения для прочности: ", results_s.best_params_)

In [ ]:
model_svr = SVR(kernel='sigmoid', C=0.001)

In [ ]:
create_modulus_model(model_svr, "SVR")

In [ ]:
create_strength_model(model_svr, "SVR")

Отсортируем таблицу по результатам коэффициентов детерминации (в тестовой выборке) для обеих целевых переменных:

In [ ]:
ms_table_3.sort_values(['R²m test', 'R²s test'], ascending=False)

Также выведем на экран две предыдущие таблицы для сравнения:

In [ ]:
ms_table.sort_values(['R²m test', 'R²s test'], ascending=False)

In [ ]:
ms_table_2.sort_values(['R²m test', 'R²s test'], ascending=False)

***Вывод:***    
В сравнении с предыдущими гипотезами, модель KNeighborsRegressor неплохо себя показала. Но всё также далеко от приемлемых результатов.

**ОБНОВЛЕНИЕ:** сохраним модель GradientBoostingRegressor для модуля упругости, так как в конце исследования становится ясно, что это лучшая модель по показателям.

In [ ]:
pickle.dump({'model': model_gb_m, 'scaler': scaler}, open("./models/modulus_model_with_scaler.pkl", 'wb'))

In [ ]:
class ModulusModelWithScaler:
    def __init__(self, model, scaler):
        self.model = model
        self.scaler = scaler

    def predict(self, X):
        X_full = X.copy()
        
        missing_columns = ['Соотношение матрица-наполнитель', 'Плотность, кг/м3', 
                           'Количество отвердителя, м.%', 'Содержание эпоксидных групп,%_2', 
                           'Температура вспышки, С_2', 'Поверхностная плотность, г/м2', 
                           'Модуль упругости при растяжении, ГПа', 'Прочность при растяжении, МПа', 
                           'Шаг нашивки', 'Плотность нашивки']
        for col in missing_columns:
            if col not in X_full.columns:
                X_full[col] = 0

        X_full = X_full[self.scaler.feature_names_in_]
        X_denorm = self.scaler.inverse_transform(X_full)
        X_denorm_df = pd.DataFrame(X_denorm, columns=X_full.columns)
        
        X_denorm_df = X_denorm_df.drop(columns=missing_columns, errors='ignore')

        y_pred = self.model.predict(X_denorm_df)
        return y_pred.flatten()

In [ ]:
loaded_data = pickle.load(open("./models/modulus_model_with_scaler.pkl", 'rb'))

In [ ]:
modulus_model = ModulusModelWithScaler(loaded_data['model'], loaded_data['scaler'])

In [ ]:
prediction = modulus_model.predict(X_test)

In [ ]:
prediction[0]

*Модель сохранена, но не уверена, что верно получилось денормализовать данные.*

# Вывод по сделанной работе

При реальной работе с подобными таблицами, я бы однозначно обращалась за дополнительными данными к заказчику, там как их недостаточно по количеству, а также связи между признаками и целевыми переменными практически нет.

***Модуль упругости:***    
Лучшая модель для прогноза модуля упругости - `GradientBoostingRegressor` из *третьей* гипотезы, так как эта модель показала наибольший R² (0.004) при низком MSE (0.036) на тестовых данных.

***Прочность при растяжении:***    
Лучшая модель для прогноза прочности при растяжении - `LinearRegression` из *второй* гипотезы, так как эта модель показала наибольший R² (0.024) при низком MSE (0.026) на тестовых данных.

Впрочем, ни одна из получившихся моделей не может быть использована для точных пресказаний и не демонстрируют надёжные результаты.

Несмотря на это, считаю поставленную задачу выполненной.    
Спасибо за внимание!